In [0]:
CREATE TABLE IF NOT EXISTS workspace.myra_invest.gold_stock_recommendations (

    symbol STRING,
    companyName STRING,

    tradeDate DATE,

    open DOUBLE,
    high DOUBLE,
    low DOUBLE,
    close DOUBLE,
    volume BIGINT,

    movingAvg5 DOUBLE,
    movingAvg20 DOUBLE,

    priceChange DOUBLE,
    priceChangePct DOUBLE,
    dailyReturnPct DOUBLE,

    source STRING,
    ingestionTimestamp TIMESTAMP,
    ingestionDate DATE

)
USING DELTA;

In [0]:
%python
from pyspark.sql.functions import *

silver_df = spark.table("workspace.myra_invest.silver_stock_prices")

print(f"Silver records: {silver_df.count()}")

display(silver_df.limit(5))

gold_df = (
    silver_df.select(
        "symbol",
        "companyName",
        "tradeDate",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "movingAvg5",
        "movingAvg20",
        "priceChange",
        "priceChangePct",
        "dailyReturnPct",
        "source",
        "ingestionTimestamp",
        "ingestionDate"
    )
)

display(gold_df.limit(10))

In [0]:
%python
from pyspark.sql.functions import *

silver_df = spark.table("workspace.myra_invest.silver_stock_prices")

print(f"Silver records: {silver_df.count()}")

display(silver_df.limit(5))

In [0]:
%python
gold_df = (
    silver_df.select(
        "symbol",
        "companyName",
        "tradeDate",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "movingAvg5",
        "movingAvg20",
        "priceChange",
        "priceChangePct",
        "dailyReturnPct",
        "source",
        "ingestionTimestamp",
        "ingestionDate"
    )
)

display(gold_df.limit(10))

In [0]:
DESCRIBE TABLE workspace.myra_invest.silver_stock_prices;

In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.window import Window

bronze_df = spark.table("workspace.myra_invest.bronze_stock_prices")

window5 = Window.partitionBy("symbol").orderBy("tradeDate").rowsBetween(-4, 0)
window20 = Window.partitionBy("symbol").orderBy("tradeDate").rowsBetween(-19, 0)

silver_df = (
    bronze_df
    .withColumn(
        "priceChange",
        round(col("close") - col("open"), 2)
    )
    .withColumn(
        "priceChangePct",
        round(
            when(col("open") != 0,
                 (col("close") - col("open")) / col("open") * 100
            ).otherwise(None),
            2
        )
    )
    .withColumn(
        "dailyReturnPct",
        round(
            when(col("open") != 0,
                 (col("close") - col("open")) / col("open") * 100
            ).otherwise(None),
            2
        )
    )
    .withColumn(
        "movingAvg5",
        round(avg("close").over(window5), 2)
    )
    .withColumn(
        "movingAvg20",
        round(avg("close").over(window20), 2)
    )
    .withColumn(
        "ingestionTimestamp",
        current_timestamp()
    )
    .withColumn(
        "ingestionDate",
        current_date()
    )
)

display(silver_df.limit(10))

In [0]:
%python
(
    silver_df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("workspace.myra_invest.silver_stock_prices")
)

print("✅ Silver table recreated successfully.")